# LeetCode 1864 — Minimum Number of Swaps to Make the Binary String Alternating bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-swaps-to-make-the-binary-string-alternating/ · Độ khó: **Medium** · Chủ đề: String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minSwaps(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra chuỗi xen kẽ độ dài `n` chỉ có **hai** dạng: `0101…` và `1010…` — bài toán trở thành "biến `s` thành một chuỗi đích **cố định**".
2. Điều kiện **khả thi** dựa trên số lượng ký tự: `|#0 − #1| ≤ 1`, và với `n` lẻ thì ký tự nhiều hơn phải đứng ở vị trí chẵn.
3. Chứng minh: khi số lượng khớp, số **vị trí sai** luôn chẵn, chia đều cho hai loại (`1` ở chỗ của `0` và ngược lại), và đáp án = **số vị trí sai / 2** (mỗi phép đổi sửa đúng hai chỗ).
4. Phân biệt bài này (đổi chỗ **bất kỳ**) với biến thể đổi chỗ **kề nhau** (đáp án khác hẳn).
5. Kiểm chứng bằng **BFS** trên không gian trạng thái với chuỗi ngắn.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `s`, trả về **số phép đổi chỗ ký tự ít nhất** để `s` trở thành **xen kẽ**, hoặc `−1` nếu không thể.

Chuỗi gọi là **xen kẽ** nếu không có hai ký tự kề nhau bằng nhau (ví dụ `"010"`, `"1010"` xen kẽ; `"0100"` thì không). Được đổi chỗ **hai ký tự bất kỳ**, kể cả không kề nhau.

**Ví dụ 1:** `s = "111000"` → `1` (đổi vị trí 1 và 4: `"111000" → "101010"`).

**Ví dụ 2:** `s = "010"` → `0` (đã xen kẽ).

**Ví dụ 3:** `s = "1110"` → `−1`.

**Ràng buộc:** `1 ≤ s.length ≤ 1000`; `s[i]` là `'0'` hoặc `'1'`.

In [1]:
import random
import time
from collections import deque
from itertools import product
from typing import List

examples = [("111000", 1), ("010", 0), ("1110", -1)]
for s, exp in examples:
    print(f"s={s!r} -> kỳ vọng {exp}")

s='111000' -> kỳ vọng 1
s='010' -> kỳ vọng 0
s='1110' -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Hai chuỗi xen kẽ độ dài `n`

Chuỗi xen kẽ được quyết định hoàn toàn bởi ký tự đầu: `P0 = "0101…"` và `P1 = "1010…"`. Ở `P0`, vị trí **chẵn** là `0`; ở `P1`, vị trí chẵn là `1`.

In [2]:
def pattern(n, first):
    return "".join(str((first + i) % 2) for i in range(n))

for n in (4, 5):
    print(n, pattern(n, 0), pattern(n, 1), "| số 0/1 trong P0:", pattern(n, 0).count("0"), pattern(n, 0).count("1"))

4 0101 1010 | số 0/1 trong P0: 2 2
5 01010 10101 | số 0/1 trong P0: 3 2


### 3.2 Đổi chỗ không làm thay đổi số lượng ký tự

Phép đổi chỗ chỉ hoán vị các ký tự ⇒ `#0` và `#1` của `s` bất biến. Vì vậy `s` chỉ biến được thành đích `P` nếu `#0(s) = #0(P)` và `#1(s) = #1(P)`.

- `n` chẵn: `P0` và `P1` đều có `n/2` số `0` ⇒ cần `#0 = #1`.
- `n` lẻ: `P0` có nhiều `0` hơn một, `P1` có nhiều `1` hơn một ⇒ đích duy nhất là chuỗi bắt đầu bằng ký tự **nhiều hơn**.

### 3.3 Đếm vị trí sai

So `s` với đích `P`: gọi `a` = số vị trí `s[i] = '1'` nhưng `P[i] = '0'`, `b` = số vị trí `s[i] = '0'` nhưng `P[i] = '1'`. Nếu số lượng khớp thì `a = b`.

In [3]:
s = "111000"; P = pattern(len(s), 1)
a = sum(x == "1" and y == "0" for x, y in zip(s, P)); b = sum(x == "0" and y == "1" for x, y in zip(s, P))
print(f"s={s}, P={P}: a={a} (1 thừa), b={b} (0 thừa)")

s=111000, P=101010: a=1 (1 thừa), b=1 (0 thừa)


### 3.4 BFS cho "số phép biến đổi ít nhất"

Khi mỗi phép biến đổi có chi phí 1, **BFS** từ trạng thái đầu cho khoảng cách ngắn nhất tới mọi trạng thái — dùng làm brute force đáng tin cậy với `n` nhỏ.

### 3.5 Bất biến: thử nghiệm với các phép đổi chỗ ngẫu nhiên

Thực hiện hàng nghìn phép đổi chỗ ngẫu nhiên trên một chuỗi: số `'0'` và `'1'` không bao giờ đổi, nên tập các chuỗi **đạt được** đúng bằng tập các hoán vị của `s`. Đây là lý do điều kiện khả thi chỉ phụ thuộc vào số lượng.

In [4]:
def is_alt(s):
    return all(s[i] != s[i + 1] for i in range(len(s) - 1))

rng = random.Random(35)
s0 = "1110001"; a = list(s0); seen = set()
for _ in range(5000):
    i, j = rng.randrange(len(a)), rng.randrange(len(a))
    a[i], a[j] = a[j], a[i]
    seen.add("".join(a))
    assert a.count("1") == s0.count("1")
from math import comb
print(f"Số chuỗi khác nhau gặp được: {len(seen)} = C(7, 4) = {comb(7, 4)}; số '1' luôn là {s0.count('1')}")
print("Chuỗi xen kẽ nằm trong số đó:", sorted(x for x in seen if is_alt(x)))

Số chuỗi khác nhau gặp được: 35 = C(7, 4) = 35; số '1' luôn là 4
Chuỗi xen kẽ nằm trong số đó: ['1010101']


## 4. Ý tưởng nền tảng / trực giác

Vì chỉ có hai đích khả dĩ, ta tính "chi phí biến `s` thành `P0`" và "thành `P1`", lấy nhỏ hơn (trong số đích khả thi).

Chi phí biến `s` thành `P` cố định: mỗi vị trí sai là "`1` nằm chỗ `0`" hoặc "`0` nằm chỗ `1`". Một phép đổi chỗ giữa một vị trí loại thứ nhất và một vị trí loại thứ hai sửa **cả hai** cùng lúc — tốt nhất có thể. Vậy chi phí = `a = b = (số vị trí sai) / 2`.

Tính tay **Ví dụ 1** `"111000"` (`n = 6`, `#0 = #1 = 3` ⇒ cả hai đích khả thi):

| đích | so sánh | số vị trí sai | chi phí |
|---|---|---|---|
| `P0 = 010101` | `111000` vs `010101` | vị trí 0, 2, 3, 5 ⇒ 4 | 2 |
| `P1 = 101010` | `111000` vs `101010` | vị trí 1, 4 ⇒ 2 | **1** |

**Ví dụ 3** `"1110"`: `#1 = 3`, `#0 = 1`, chênh 2 ⇒ `−1`.

In [5]:
for s, _ in examples:
    n, c0, c1 = len(s), s.count("0"), s.count("1")
    for first in (0, 1):
        P = pattern(n, first)
        feasible = P.count("0") == c0
        mism = sum(x != y for x, y in zip(s, P))
        print(f"s={s:<7} P={P:<7} khả thi={feasible!s:<5} sai={mism} -> {mism // 2 if feasible else '—'}")

s=111000  P=010101  khả thi=True  sai=4 -> 2
s=111000  P=101010  khả thi=True  sai=2 -> 1
s=010     P=010     khả thi=True  sai=0 -> 0
s=010     P=101     khả thi=False sai=3 -> —
s=1110    P=0101    khả thi=False sai=3 -> —
s=1110    P=1010    khả thi=False sai=1 -> —


## 5. Cách 1 — Brute Force / baseline: BFS trên các trạng thái

Mỗi trạng thái là một chuỗi; từ mỗi trạng thái, thử đổi chỗ mọi cặp vị trí có ký tự **khác nhau** (đổi hai ký tự giống nhau là vô ích). Dừng khi gặp chuỗi xen kẽ. Nếu BFS duyệt hết mà không gặp ⇒ `−1`.

In [6]:
def is_alt(s):
    return all(s[i] != s[i + 1] for i in range(len(s) - 1))

def minswaps_bfs(s):
    if is_alt(s):
        return 0
    dist = {s: 0}; q = deque([s])
    while q:
        cur = q.popleft(); arr = list(cur)
        for i in range(len(arr)):
            for j in range(i + 1, len(arr)):
                if arr[i] != arr[j]:
                    arr[i], arr[j] = arr[j], arr[i]
                    nxt = "".join(arr)
                    arr[i], arr[j] = arr[j], arr[i]
                    if nxt not in dist:
                        dist[nxt] = dist[cur] + 1
                        if is_alt(nxt):
                            return dist[nxt]
                        q.append(nxt)
    return -1

for s, exp in examples:
    got = minswaps_bfs(s); print(s, "->", got, "kỳ vọng", exp); assert got == exp

111000 -> 1 kỳ vọng 1
010 -> 0 kỳ vọng 0
1110 -> -1 kỳ vọng -1


**Độ phức tạp:** số trạng thái = `C(n, #1)` (mọi hoán vị của cùng đa tập), mỗi trạng thái `O(n²)` láng giềng ⇒ chỉ dùng được với `n ≲ 12`. Với `n = 1000` là không tưởng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (hai đích):** đáp án `= min` chi phí tới `P0`, `P1` (chỉ xét đích có cùng số lượng ký tự với `s`).

**Insight 2 (chi phí tới một đích cố định):** với đích `P` khả thi, số vị trí "`1` thừa" bằng số vị trí "`0` thừa" (`a = b`), và chi phí tối thiểu đúng bằng `a`:
- **Cận dưới:** mỗi phép đổi chỗ thay đổi tối đa 2 vị trí ⇒ sửa tối đa 2 vị trí sai ⇒ cần ít nhất `(a + b)/2 = a` phép.
- **Đạt được:** ghép từng vị trí "`1` thừa" với một vị trí "`0` thừa" và đổi chỗ ⇒ đúng `a` phép.

**Insight 3 (đếm gọn):** với `P0`, các vị trí "`1` thừa" chính là các vị trí **chẵn** chứa `'1'`. Gọi `e1` = số `'1'` ở vị trí chẵn và `E` = số vị trí chẵn. Khi đích tương ứng khả thi: chi phí tới `P0` là `e1`, chi phí tới `P1` là `E − e1` (số `'0'` ở vị trí chẵn).

### 6.1 Minh họa ghép cặp vị trí sai

Với `s = "0110100110"` (ví dụ tự dựng, đối chiếu lại đề gốc) và đích `P1 = "1010101010"`, liệt kê hai tập `A` ("`1` thừa") và `B` ("`0` thừa"). Hai tập luôn cùng kích thước; ghép `A[k]` với `B[k]` cho một lời giải tối ưu. Thứ tự ghép không quan trọng — mọi cách ghép đều cho cùng số phép.

In [7]:
s = "0110100110"; P = pattern(len(s), 1)
A = [i for i in range(len(s)) if s[i] == "1" and P[i] == "0"]
B = [i for i in range(len(s)) if s[i] == "0" and P[i] == "1"]
print("s =", s, "\nP =", P, "\nA =", A, "B =", B)
for perm in ([*zip(A, B)], [*zip(A, B[::-1])]):
    r = list(s)
    for i, j in perm:
        r[i], r[j] = r[j], r[i]
    print("ghép", perm, "->", "".join(r), "xen kẽ:", is_alt("".join(r)))

s = 0110100110 
P = 1010101010 
A = [1, 7] B = [0, 6]
ghép [(1, 0), (7, 6)] -> 1010101010 xen kẽ: True
ghép [(1, 6), (7, 0)] -> 1010101010 xen kẽ: True


## 7. Các cách tiếp theo

### 7.1 Cách 2 — So với hai chuỗi đích, đếm vị trí sai

In [8]:
def minswaps_patterns(s):
    n, c0 = len(s), s.count("0")
    best = -1
    for first in (0, 1):
        P = pattern(n, first)
        if P.count("0") != c0:
            continue                                     # đích không khả thi
        cost = sum(x != y for x, y in zip(s, P)) // 2
        best = cost if best == -1 else min(best, cost)
    return best

print([minswaps_patterns(s) for s, _ in examples])

[1, 0, -1]


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(n)` bộ nhớ cho chuỗi đích (có thể bỏ, xem Cách 3).

### 7.2 Cách 3 — Công thức đếm một lượt

Chỉ cần ba số: `n`, `c1 = #1`, `e1 = #'1'` ở vị trí chẵn. Số vị trí chẵn là `E = ⌈n/2⌉`.
- Tới `P0` (vị trí chẵn phải là `0`): chi phí `e1`; khả thi khi `c1 = ⌊n/2⌋`.
- Tới `P1` (vị trí chẵn phải là `1`): chi phí `E − e1` (số `'0'` ở vị trí chẵn); khả thi khi `c1 = E`.

In [9]:
class Solution:
    def minSwaps(self, s: str) -> int:
        n = len(s)
        c1 = s.count("1")
        e1 = s[::2].count("1")                    # số '1' ở vị trí chẵn
        E = (n + 1) // 2                          # số vị trí chẵn
        best = []
        if c1 == n // 2:
            best.append(e1)                       # đích 0101...
        if c1 == E:
            best.append(E - e1)                   # đích 1010...
        return min(best) if best else -1

for s, exp in examples:
    got = Solution().minSwaps(s); print(s, "->", got); assert got == exp

111000 -> 1
010 -> 0
1110 -> -1


Với `n` chẵn, `n // 2 = E` nên cả hai điều kiện cùng đúng hoặc cùng sai; với `n` lẻ, tối đa một điều kiện đúng — khớp mục 3.2.

**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(n)` cho lát cắt `s[::2]` (hoặc `O(1)` nếu đếm bằng vòng lặp).

### 7.3 Cách 4 — Xây dựng cụ thể các phép đổi chỗ (constructive)

Không chỉ đếm, ta liệt kê các cặp `(i, j)` cần đổi: gom danh sách vị trí "`1` thừa" và "`0` thừa" rồi ghép đôi. Hữu ích để **kiểm chứng** đáp án bằng cách thực hiện thật.

In [10]:
def swap_plan(s):
    n, c0 = len(s), s.count("0")
    best = None
    for first in (0, 1):
        P = pattern(n, first)
        if P.count("0") != c0:
            continue
        ones = [i for i in range(n) if s[i] == "1" and P[i] == "0"]
        zeros = [i for i in range(n) if s[i] == "0" and P[i] == "1"]
        assert len(ones) == len(zeros)
        plan = list(zip(ones, zeros))
        if best is None or len(plan) < len(best):
            best = plan
    return best                                          # None nếu không khả thi

def apply_plan(s, plan):
    a = list(s)
    for i, j in plan:
        a[i], a[j] = a[j], a[i]
    return "".join(a)

for s, _ in examples:
    plan = swap_plan(s)
    print(s, "->", plan, "->", apply_plan(s, plan) if plan is not None else "không thể")

111000 -> [(1, 4)] -> 101010
010 -> [] -> 010
1110 -> None -> không thể


**Độ phức tạp Cách 4:** `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên ba ví dụ và một ví dụ lẻ tự dựng `"11001"` (đối chiếu lại đề gốc):

In [11]:
def trace_formula(s):
    n, c1 = len(s), s.count("1")
    e1, E = s[::2].count("1"), (n + 1) // 2
    print(f"s={s!r}: n={n}, c1={c1}, vị trí chẵn s[::2]={s[::2]!r} -> e1={e1}, E={E}")
    opts = []
    if c1 == n // 2:
        opts.append(e1); print(f"   đích 0101.. khả thi (c1 = n//2 = {n // 2}), chi phí e1 = {e1}")
    else:
        print(f"   đích 0101.. KHÔNG khả thi (cần c1 = {n // 2})")
    if c1 == E:
        opts.append(E - e1); print(f"   đích 1010.. khả thi (c1 = E = {E}), chi phí E - e1 = {E - e1}")
    else:
        print(f"   đích 1010.. KHÔNG khả thi (cần c1 = {E})")
    ans = min(opts) if opts else -1
    print("   =>", ans)
    return ans

for s, exp in examples:
    assert trace_formula(s) == exp
assert trace_formula("11001") == minswaps_bfs("11001")

s='111000': n=6, c1=3, vị trí chẵn s[::2]='110' -> e1=2, E=3
   đích 0101.. khả thi (c1 = n//2 = 3), chi phí e1 = 2
   đích 1010.. khả thi (c1 = E = 3), chi phí E - e1 = 1
   => 1
s='010': n=3, c1=1, vị trí chẵn s[::2]='00' -> e1=0, E=2
   đích 0101.. khả thi (c1 = n//2 = 1), chi phí e1 = 0
   đích 1010.. KHÔNG khả thi (cần c1 = 2)
   => 0
s='1110': n=4, c1=3, vị trí chẵn s[::2]='11' -> e1=2, E=2
   đích 0101.. KHÔNG khả thi (cần c1 = 2)
   đích 1010.. KHÔNG khả thi (cần c1 = 2)
   => -1
s='11001': n=5, c1=3, vị trí chẵn s[::2]='101' -> e1=2, E=3
   đích 0101.. KHÔNG khả thi (cần c1 = 2)
   đích 1010.. khả thi (c1 = E = 3), chi phí E - e1 = 1
   => 1


| s | n | c1 | e1 | E | đích khả thi | chi phí | đáp án |
|---|---|---|---|---|---|---|---|
| `111000` | 6 | 3 | 2 | 3 | cả hai | `2`, `1` | **1** |
| `010` | 3 | 1 | 0 | 2 | `0101..` | `0` | **0** |
| `1110` | 4 | 3 | 2 | 2 | không | — | **−1** |
| `11001` | 5 | 3 | 2 | 3 | `1010..` | `3 − 2 = 1` | **1** |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — đếm `c1`, `e1` rồi áp công thức.

In [12]:
class Solution:
    def minSwaps(self, s: str) -> int:
        n = len(s)
        c1 = s.count("1")
        e1 = s[::2].count("1")
        E = (n + 1) // 2
        options = []
        if c1 == n // 2:
            options.append(e1)
        if c1 == E:
            options.append(E - e1)
        return min(options) if options else -1

for s, exp in examples:
    got = Solution().minSwaps(s); print(got, exp); assert got == exp

1 1
0 0
-1 -1


## 10. Vì sao thuật toán đúng

**Khả thi.** Đổi chỗ bảo toàn số lượng ký tự; một chuỗi xen kẽ độ dài `n` là `P0` hoặc `P1`, với số `'1'` lần lượt `⌊n/2⌋` và `⌈n/2⌉`. Vậy có thể ⇔ `c1 ∈ {⌊n/2⌋, ⌈n/2⌉}` tương ứng đích.

**Chi phí tới đích `P` khả thi.** Gọi `A` = tập vị trí có `s[i] = 1, P[i] = 0`, `B` = tập vị trí có `s[i] = 0, P[i] = 1`. Vì số `'1'` của `s` và `P` bằng nhau: `#1(s) = #1 ở vị trí đúng + |A|`, `#1(P) = #1 ở vị trí đúng + |B|` ⇒ `|A| = |B|`.
- *Cận dưới:* một phép đổi chỗ chỉ thay đổi hai vị trí ⇒ giảm số vị trí sai tối đa 2 ⇒ cần `≥ (|A| + |B|)/2 = |A|` phép.
- *Đạt được:* đổi chỗ từng cặp `(a ∈ A, b ∈ B)` sửa đúng hai vị trí ⇒ `|A|` phép.

**Công thức.** Với `P0`, `A` = vị trí chẵn chứa `'1'` ⇒ `|A| = e1`. Với `P1`, `B` = vị trí chẵn chứa `'0'` ⇒ `|B| = E − e1`. Đáp án là `min` trên các đích khả thi. ∎

### 10.1 Kiểm chứng thực nghiệm: cận dưới đạt được và BFS khớp với mọi chuỗi ngắn

Duyệt **mọi** chuỗi nhị phân độ dài `1..8`: so BFS với công thức, và thực hiện kế hoạch của Cách 4 để chắc chắn kết quả xen kẽ với đúng số phép.

In [13]:
cnt = 0
for n in range(1, 9):
    for bits in product("01", repeat=n):
        s = "".join(bits)
        ans = Solution().minSwaps(s)
        assert ans == minswaps_bfs(s), s
        plan = swap_plan(s)
        if ans == -1:
            assert plan is None
        else:
            assert len(plan) == ans and is_alt(apply_plan(s, plan))
        cnt += 1
print(f"Đã kiểm {cnt} chuỗi (mọi chuỗi độ dài 1..8): BFS = công thức = độ dài kế hoạch")

Đã kiểm 510 chuỗi (mọi chuỗi độ dài 1..8): BFS = công thức = độ dài kế hoạch


## 11. Phân tích Time / Space Complexity

Gọi `n = len(s) ≤ 1000`.

| Cách | Time | Space |
|---|---|---|
| 1. BFS trạng thái | `O(C(n, c1) · n²)` | `O(C(n, c1))` |
| 2. So với hai đích | `O(n)` | `O(n)` |
| 3. Công thức | `O(n)` | `O(1)` (bỏ lát cắt) |
| 4. Kế hoạch đổi chỗ | `O(n)` | `O(n)` |

In [14]:
for n in (8, 10, 12, 14, 16):
    s = "1" * (n // 2 + 1) + "0" * (n // 2 - 1)          # không khả thi: BFS phải duyệt HẾT trạng thái
    t = time.perf_counter(); r = minswaps_bfs(s); print(f"BFS n={n:>2} (không khả thi): {r} ({(time.perf_counter() - t) * 1000:.0f} ms)")
big = "1" * 500 + "0" * 500
t = time.perf_counter()
for _ in range(1000):
    Solution().minSwaps(big)
print(f"Công thức n=1000 x1000 lần: {(time.perf_counter() - t) * 1000:.0f} ms, đáp án {Solution().minSwaps(big)}")

BFS n= 8 (không khả thi): -1 (0 ms)
BFS n=10 (không khả thi): -1 (1 ms)
BFS n=12 (không khả thi): -1 (6 ms)
BFS n=14 (không khả thi): -1 (31 ms)


BFS n=16 (không khả thi): -1 (161 ms)
Công thức n=1000 x1000 lần: 1 ms, đáp án 250


## 12. Bộ test

In [15]:
ALL = {"patterns": minswaps_patterns, "submit": Solution().minSwaps,
       "plan": lambda s: -1 if swap_plan(s) is None else len(swap_plan(s))}
def run_all(s, brute=True):
    res = {k: f(s) for k, f in ALL.items()}
    if brute:
        res["bfs"] = minswaps_bfs(s)
    assert len(set(res.values())) == 1, (s, res)
    return res["submit"]

for s, exp in examples:
    assert run_all(s) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng BFS trong run_all
edge = [("0", 0), ("1", 0), ("00", -1), ("01", 0), ("10", 0), ("100", 1), ("001", 1),
        ("11001", 1), ("100101", 1), ("1100", 1), ("0000011111", 2)]
for s, exp in edge:
    assert run_all(s) == exp, (s, run_all(s))
rng = random.Random(1864)
for _ in range(300):
    run_all("".join(rng.choice("01") for _ in range(rng.randint(1, 10))))
for _ in range(200):
    n = rng.randint(1, 1000); ones = rng.choice([n // 2, (n + 1) // 2, rng.randint(0, n)])
    s = list("1" * ones + "0" * (n - ones)); rng.shuffle(s)
    run_all("".join(s), brute=False)
print("Ví dụ, 11 edge case, 300 chuỗi độ dài ≤ 10 (BFS), 200 chuỗi dài tới 1000: OK")

Ví dụ, 11 edge case, 300 chuỗi độ dài ≤ 10 (BFS), 200 chuỗi dài tới 1000: OK


### 12.1 Kiểm tra tính chất

- Đảo bit (`0 ↔ 1`) không đổi đáp án (đổi vai `P0`, `P1`).
- Đảo ngược chuỗi không đổi đáp án: với `n` chẵn, đảo ngược biến `P0` thành `P1` (hai chi phí đổi chỗ cho nhau); với `n` lẻ, đích duy nhất là chuỗi đối xứng.
- Đáp án `≤ ⌊n/2⌋`, và bằng 0 ⇔ `s` đã xen kẽ.

In [16]:
rng = random.Random(121)
f = Solution().minSwaps
flip = str.maketrans("01", "10")
for _ in range(2000):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 30)))
    a = f(s)
    assert f(s.translate(flip)) == a and f(s[::-1]) == a
    assert a <= len(s) // 2
    assert (a == 0) == is_alt(s)
print("Tính chất đảo bit, đảo chuỗi, cận trên, a == 0 <=> xen kẽ: đúng trên 2000 chuỗi")

Tính chất đảo bit, đảo chuỗi, cận trên, a == 0 <=> xen kẽ: đúng trên 2000 chuỗi


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không kiểm tra khả thi theo số lượng

In [17]:
def wrong_no_feasibility(s):
    n = len(s)
    return min(sum(x != y for x, y in zip(s, pattern(n, f))) // 2 for f in (0, 1))   # SAI

for s, exp in examples:
    print(f"{s}: sai {wrong_no_feasibility(s)} | đúng {exp}")

111000: sai 1 | đúng 1
010: sai 0 | đúng 0
1110: sai 0 | đúng -1


**Vì sao sai:** với `"1110"` không thể đạt chuỗi xen kẽ, nhưng số vị trí sai tới `1010` là 1 ⇒ `1 // 2 = 0`. **Sửa:** chỉ xét đích có cùng số `'1'` với `s`.

### Lỗi 2 — Quên chia 2 (đếm số vị trí sai thay vì số phép đổi)

In [18]:
def wrong_no_half(s):
    n, c0 = len(s), s.count("0")
    opts = [sum(x != y for x, y in zip(s, pattern(n, f))) for f in (0, 1) if pattern(n, f).count("0") == c0]
    return min(opts) if opts else -1                                                    # SAI

for s, exp in examples:
    print(f"{s}: sai {wrong_no_half(s)} | đúng {exp}")

111000: sai 2 | đúng 1
010: sai 0 | đúng 0
1110: sai -1 | đúng -1


**Vì sao sai:** mỗi phép đổi chỗ sửa **hai** vị trí sai. **Sửa:** `// 2`.

### Lỗi 3 — `n` lẻ: lấy `min` trên cả hai đích, kể cả đích không khả thi

In [19]:
def wrong_odd(s):
    n = len(s)
    if abs(s.count("0") - s.count("1")) > 1:
        return -1                                            # kiểm tra chênh lệch, nhưng...
    return min(sum(x != y for x, y in zip(s, pattern(n, f))) // 2 for f in (0, 1))   # SAI: n lẻ chỉ có 1 đích

s = "100"          # ví dụ tự dựng (đối chiếu lại đề gốc)
print(f"{s}: sai {wrong_odd(s)} | đúng {Solution().minSwaps(s)} | BFS {minswaps_bfs(s)}")

100: sai 0 | đúng 1 | BFS 1


**Vì sao sai:** `"100"` có hai `0`, một `1` ⇒ đích duy nhất `"010"` (2 vị trí sai ⇒ 1 phép). Đích `"101"` cần hai `'1'` — không khả thi — nhưng chỉ có 1 vị trí sai ⇒ `1 // 2 = 0`. **Sửa:** `n` lẻ thì ký tự nhiều hơn phải đứng ở vị trí chẵn.

### Lỗi 4 — Luôn chọn đích bắt đầu bằng `s[0]`

In [20]:
def wrong_first_char(s):
    n = len(s)
    if abs(s.count("0") - s.count("1")) > 1:
        return -1
    first = int(s[0]) if n % 2 == 0 else int(s.count("1") > s.count("0"))
    return sum(x != y for x, y in zip(s, pattern(n, first))) // 2                    # SAI khi n chẵn

s = "100101"       # ví dụ tự dựng (đối chiếu lại đề gốc)
print(f"{s}: sai {wrong_first_char(s)} | đúng {Solution().minSwaps(s)} | BFS {minswaps_bfs(s)}")

100101: sai 2 | đúng 1 | BFS 1


**Vì sao sai:** giữ nguyên ký tự đầu không phải lúc nào cũng rẻ nhất: `"100101"` tới `101010` cần 2 phép, tới `010101` chỉ 1 phép (đổi vị trí 0 và 1). **Sửa:** `n` chẵn thì thử **cả hai** đích.

### Lỗi 5 — Nhầm với bài đổi chỗ **kề nhau**

In [21]:
def adjacent_swaps(s):
    # số phép đổi chỗ kề nhau tối thiểu (bài khác!): ghép '1' thứ k với vị trí '1' thứ k của đích
    n, best = len(s), None
    ones = [i for i, c in enumerate(s) if c == "1"]
    for f in (0, 1):
        P = pattern(n, f)
        tgt = [i for i, c in enumerate(P) if c == "1"]
        if len(tgt) == len(ones):
            cost = sum(abs(a - b) for a, b in zip(ones, tgt))
            best = cost if best is None else min(best, cost)
    return -1 if best is None else best

for s, exp in examples:
    print(f"{s}: kề nhau {adjacent_swaps(s)} | đề này (bất kỳ) {exp}")

111000: kề nhau 3 | đề này (bất kỳ) 1
010: kề nhau 0 | đề này (bất kỳ) 0
1110: kề nhau -1 | đề này (bất kỳ) -1


**Vì sao sai:** đề cho đổi chỗ **bất kỳ** hai ký tự; số phép ít hơn nhiều so với đổi kề nhau (`"111000"`: 1 so với 3). Đọc kỹ "even if they are not adjacent". Biến thể kề nhau xem mục 15.

### Lỗi 6 — Dùng `s.count('1') == len(s) // 2` cho cả hai đích khi `n` lẻ

In [22]:
def wrong_parity(s):
    n, c1 = len(s), s.count("1")
    e1, E = s[::2].count("1"), (n + 1) // 2
    opts = []
    if c1 == n // 2: opts.append(e1)
    if c1 == n // 2: opts.append(E - e1)                         # SAI: phải là c1 == E
    return min(opts) if opts else -1

s = "11001"        # ví dụ tự dựng (đối chiếu lại đề gốc)
print(f"{s}: sai {wrong_parity(s)} | đúng {Solution().minSwaps(s)} | BFS {minswaps_bfs(s)}")

11001: sai -1 | đúng 1 | BFS 1


**Vì sao sai:** với `n` lẻ, `⌊n/2⌋ ≠ ⌈n/2⌉`; đích `1010…1` có `⌈n/2⌉` số `'1'`. Nhầm điều kiện khiến chuỗi khả thi bị trả `−1` (hoặc ngược lại). **Sửa:** điều kiện riêng cho mỗi đích.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Biến thành chuỗi **xen kẽ** / dạng cố định" ⇒ liệt kê hết các **đích** (thường 2) và tính chi phí tới từng đích.
- "Đổi chỗ **bất kỳ**" ⇒ chi phí = số vị trí sai / 2 (mỗi phép sửa 2 chỗ), cần số lượng khớp.
- "Đổi chỗ **kề nhau**" ⇒ tổng khoảng cách ghép đôi theo thứ tự (hoặc đếm nghịch thế).
- "Lật bit" (flip) thay vì đổi chỗ ⇒ chi phí = số vị trí sai (bài 1758).
- Luôn nghĩ đến bất biến: phép biến đổi giữ nguyên gì? (ở đây: số lượng mỗi ký tự).

## 15. Biến thể của bài

1. **1758 Minimum Changes To Make Alternating Binary String**: **lật** ký tự ⇒ `min(sai tới P0, sai tới P1)`, luôn khả thi.
2. **Đổi chỗ kề nhau** để thành xen kẽ: tổng `|vị trí '1' thứ k − vị trí đích thứ k|` (code ở Lỗi 5).
3. **1888 Minimum Number of Flips to Make the Binary String Alternating**: cho phép xoay vòng + lật — cửa sổ trượt trên `s + s`.
4. **Bảng `n × n` xen kẽ (bàn cờ) bằng đổi hàng/cột** — 782 Transform to Chessboard (Hard), cùng ý tưởng đếm vị trí sai / 2 trên từng chiều.
5. **Chuỗi ba ký tự** "không hai ký tự kề nhau bằng nhau" — không còn chỉ 2 đích; bài khó hơn.

### 15.1 Code biến thể: 1758 (lật) và đổi chỗ kề nhau — đối chiếu brute force

In [23]:
def min_flips_alt(s):                                     # 1758
    n = len(s)
    return min(sum(x != y for x, y in zip(s, pattern(n, f))) for f in (0, 1))

def adjacent_bfs(s):
    if is_alt(s): return 0
    dist = {s: 0}; q = deque([s])
    while q:
        cur = q.popleft()
        for i in range(len(cur) - 1):
            if cur[i] != cur[i + 1]:
                nxt = cur[:i] + cur[i + 1] + cur[i] + cur[i + 2:]
                if nxt not in dist:
                    dist[nxt] = dist[cur] + 1
                    if is_alt(nxt): return dist[nxt]
                    q.append(nxt)
    return -1

for n in range(1, 10):
    for bits in product("01", repeat=n):
        s = "".join(bits)
        assert adjacent_swaps(s) == adjacent_bfs(s)
        assert min_flips_alt(s) == min(sum(x != y for x, y in zip(s, t)) for t in (pattern(n, 0), pattern(n, 1)))
print("Đổi chỗ kề nhau (công thức) = BFS kề nhau trên mọi chuỗi độ dài ≤ 9;", min_flips_alt("0100"), "(1758: '0100' -> 1)")

Đổi chỗ kề nhau (công thức) = BFS kề nhau trên mọi chuỗi độ dài ≤ 9; 1 (1758: '0100' -> 1)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | Duyệt trạng thái | mũ | mũ | Kiểm chứng `n ≤ 12` |
| 2. Hai đích | Đếm vị trí sai / 2 | `O(n)` | `O(n)` | Dễ hiểu |
| 3. Công thức | `c1`, `e1`, `E` | `O(n)` | `O(1)` | **Bản nộp** |
| 4. Kế hoạch | Ghép cặp vị trí sai | `O(n)` | `O(n)` | Cần in các phép đổi |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `"1"`; (b) `"0011"`; (c) `"00011"`; (d) `"000111"`.

**Bài 2 (tính tay):** Với `s = "0110100110"`, tính `c1`, `e1`, `E`, chi phí tới mỗi đích và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `feasible_targets(s)` trả danh sách đích khả thi (`"0101…"`/`"1010…"`).

**Bài 4 (tự viết lại):** Viết `min_swaps_to(s, t)`: số phép đổi chỗ bất kỳ ít nhất để biến chuỗi nhị phân `s` thành chuỗi nhị phân `t` cho trước (cùng độ dài), `−1` nếu không thể.

**Bài 5 (mini challenge):** Viết `min_adjacent_alt(s)` — biến thể **đổi chỗ kề nhau** — bằng công thức ghép đôi `O(n)`, và chứng minh (bằng kiểm thử với BFS) rằng ghép `'1'` thứ `k` với vị trí `'1'` thứ `k` của đích là tối ưu.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def feasible_targets(s):
    # TODO: với first in (0, 1): P = pattern(len(s), first); giữ P nếu P.count('1') == s.count('1')
    pass

In [25]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def min_swaps_to(s, t):
    # TODO: nếu số '1' khác nhau -> -1; ngược lại đếm vị trí s='1', t='0'
    pass

def min_adjacent_alt(s):
    # TODO: với mỗi đích khả thi: tổng |ones[k] - tgt[k]|; lấy min
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: kiểm tra số lượng. Mức 2: `n` chẵn thử cả hai đích. Mức 3: (a) `0`; (b) `1` (`0011 → 0101` hoặc `1010`, đều 1 phép); (c) `1` (đích `01010`, vị trí sai 1 và 4); (d) `1` (đích `010101`: sai ở vị trí 1, 4 ⇒ 1 phép; đích `101010` cần 2 phép).

**Bài 2.** Mức 1: `n = 10`, `c1 = 5` ⇒ cả hai đích. Mức 2: vị trí chẵn `s[::2] = "01101"` ⇒ `e1 = 3`, `E = 5`. Mức 3: chi phí `3` và `2` ⇒ đáp án `2`.

**Bài 3.** Mức 1: tối đa hai đích. Mức 2: so số `'1'`. Mức 3: `n` chẵn có thể cả hai hoặc không cái nào.

**Bài 4.** Mức 1: bất biến số lượng. Mức 2: đếm hai loại vị trí sai, chúng bằng nhau. Mức 3: đáp án = một trong hai số đó.

**Bài 5.** Mức 1: mỗi phép đổi kề chỉ dịch một `'1'` đi một bước. Mức 2: các `'1'` không cần "vượt" nhau ⇒ giữ thứ tự. Mức 3: tổng khoảng cách là cận dưới và đạt được.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2, 3
f = Solution().minSwaps
b1 = [f(x) for x in ("1", "0011", "00011", "000111")]
print("Bài 1:", b1); assert b1 == [0, 1, 1, 1] and b1 == [minswaps_bfs(x) for x in ("1", "0011", "00011", "000111")]
s2 = "0110100110"
print("Bài 2:"); assert trace_formula(s2) == 2
def feasible_targets(s):
    return [P for P in (pattern(len(s), 0), pattern(len(s), 1)) if P.count("1") == s.count("1")]
print("Bài 3:", feasible_targets("1100"), feasible_targets("100"), feasible_targets("1110"))
assert feasible_targets("100") == ["010"] and feasible_targets("1110") == []

Bài 1: [0, 1, 1, 1]
Bài 2:
s='0110100110': n=10, c1=5, vị trí chẵn s[::2]='01101' -> e1=3, E=5
   đích 0101.. khả thi (c1 = n//2 = 5), chi phí e1 = 3
   đích 1010.. khả thi (c1 = E = 5), chi phí E - e1 = 2
   => 2
Bài 3: ['0101', '1010'] ['010'] []


In [27]:
# Bài 4, 5
def min_swaps_to(s, t):
    if s.count("1") != t.count("1"):
        return -1
    return sum(a == "1" and b == "0" for a, b in zip(s, t))

def min_adjacent_alt(s):
    return adjacent_swaps(s)

def bfs_to(s, t):
    dist = {s: 0}; q = deque([s])
    while q:
        cur = q.popleft()
        if cur == t: return dist[cur]
        a = list(cur)
        for i in range(len(a)):
            for j in range(i + 1, len(a)):
                if a[i] != a[j]:
                    a[i], a[j] = a[j], a[i]; nxt = "".join(a); a[i], a[j] = a[j], a[i]
                    if nxt not in dist:
                        dist[nxt] = dist[cur] + 1; q.append(nxt)
    return -1

rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 8)
    s = "".join(rng.choice("01") for _ in range(n)); t = "".join(rng.choice("01") for _ in range(n))
    assert min_swaps_to(s, t) == bfs_to(s, t)
    assert min_adjacent_alt(s) == adjacent_bfs(s)
print("Bài 4, 5: OK |", min_swaps_to("1100", "0011"), min_adjacent_alt("111000"))

Bài 4, 5: OK | 2 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
n, c1 = len(s), s.count("1")
e1, E = s[::2].count("1"), (n + 1) // 2      # '1' ở vị trí chẵn, số vị trí chẵn
opts = []
if c1 == n // 2: opts.append(e1)             # đích 0101...
if c1 == E:      opts.append(E - e1)         # đích 1010...
return min(opts) if opts else -1
```

**Checklist:**
- [ ] Kiểm tra **khả thi** bằng số lượng ký tự cho **từng** đích?
- [ ] `n` lẻ: chỉ một đích (ký tự nhiều hơn đứng đầu)?
- [ ] `n` chẵn: thử **cả hai** đích?
- [ ] Chia 2 số vị trí sai (đổi chỗ sửa hai chỗ)?
- [ ] Phân biệt đổi chỗ **bất kỳ** với **kề nhau** / **lật bit**?